In [ ]:
"""
QUICKBITE PRODUCT ANALYTICS PLATFORM
Phase 2 - Notebook 6: A/B Testing & Experimentation Analysis
==================================================================
Purpose: Design, analyze, and interpret A/B tests to make data-driven
product decisions. Includes power analysis, statistical testing,
and business recommendations.

Key Questions:
1. Did the experiment show statistically significant results?
2. What is the practical significance (effect size) of the results?
3. Are guardrail metrics holding steady?
4. Should we ship, hold, or kill the experiment?

Author: Senior Product Analytics Team
Date: 2026-07-28
"""

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from scipy import stats
from scipy.stats import norm, ttest_ind, chi2_contingency, mannwhitneyu
import math
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [ ]:
print("="*80)
print("QUICKBITE A/B TESTING & EXPERIMENTATION")
print("="*80)

---------------------------------------------------------------------
1. LOAD DATA
---------------------------------------------------------------------

In [ ]:
print("\n📂 Loading data...")

In [ ]:
users = pd.read_csv('../outputs/cleaned_data/users_cleaned.csv')
orders = pd.read_csv('../outputs/cleaned_data/orders_cleaned.csv')
payments = pd.read_csv('../outputs/cleaned_data/payments_cleaned.csv')
cities = pd.read_csv('../data/cities.csv')

In [ ]:
# Convert dates
users['signup_date'] = pd.to_datetime(users['signup_date'])
orders['order_placed_at'] = pd.to_datetime(orders['order_placed_at'])
payments['processed_at'] = pd.to_datetime(payments['processed_at'])

In [ ]:
# Filter delivered orders
delivered_orders = orders[orders['order_status'] == 'delivered']

In [ ]:
print(f"✅ Loaded {len(users):,} users")
print(f"✅ Loaded {len(orders):,} orders")
print(f"✅ Loaded {len(delivered_orders):,} delivered orders")

---------------------------------------------------------------------
2. EXPERIMENT DESIGN SIMULATION
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("EXPERIMENT DESIGN SIMULATION")
print("="*80)

Create synthetic experiment data for 7 A/B tests
This simulates running the experiments from experiments_design.md

In [ ]:
def create_experiment_data(experiment_name, n_users=10000, effect_size=0.02, 
                           baseline_rate=0.28, control_ratio=0.5):
    """
    Simulate A/B test data with realistic business logic
    
    Parameters:
    - experiment_name: Name of the experiment
    - n_users: Total users in experiment
    - effect_size: Expected lift in conversion (e.g., 0.02 = 2%)
    - baseline_rate: Baseline conversion rate
    - control_ratio: % of users in control group
    """
    
    np.random.seed(42)
    
    # Assign users to control and treatment
    n_control = int(n_users * control_ratio)
    n_treatment = n_users - n_control
    
    # Create experiment assignments
    assignments = []
    
    # Control group
    for i in range(n_control):
        user_id = f"exp_user_{i}"
        variant = 'control'
        assigned_at = datetime.now() - timedelta(days=np.random.randint(1, 30))
        assignments.append({
            'user_id': user_id,
            'experiment_name': experiment_name,
            'variant': variant,
            'assigned_at': assigned_at
        })
    
    # Treatment group
    for i in range(n_treatment):
        user_id = f"exp_user_{n_control + i}"
        variant = 'treatment'
        assigned_at = datetime.now() - timedelta(days=np.random.randint(1, 30))
        assignments.append({
            'user_id': user_id,
            'experiment_name': experiment_name,
            'variant': variant,
            'assigned_at': assigned_at
        })
    
    exp_df = pd.DataFrame(assignments)
    
    # Simulate outcomes based on variant
    outcomes = []
    
    for _, row in exp_df.iterrows():
        # Base probability
        base_prob = baseline_rate
        
        # Apply treatment effect
        if row['variant'] == 'treatment':
            # Treatment effect with some variance
            treatment_effect = effect_size * (1 + np.random.normal(0, 0.1))
            conversion_prob = min(1, base_prob + treatment_effect)
        else:
            conversion_prob = base_prob
        
        # Generate outcome
        converted = np.random.random() < conversion_prob
        
        # Generate additional metrics
        order_value = 0
        cancellation = False
        payment_success = True
        
        if converted:
            # Simulate order details
            order_value = np.random.normal(350, 100)
            order_value = max(50, order_value)
            
            # Simulate cancellation (slightly higher in treatment if effect_size is positive)
            cancellation_prob = 0.08 + (0.02 if row['variant'] == 'treatment' and effect_size > 0 else 0)
            cancellation = np.random.random() < cancellation_prob
            
            # Simulate payment success
            payment_success = np.random.random() < 0.95
        
        outcomes.append({
            'user_id': row['user_id'],
            'converted': converted,
            'order_value': order_value,
            'cancelled': cancellation,
            'payment_success': payment_success
        })
    
    outcomes_df = pd.DataFrame(outcomes)
    
    # Merge assignments with outcomes
    result_df = exp_df.merge(outcomes_df, on='user_id')
    
    return result_df

In [ ]:
# Define the 7 experiments from experiments_design.md
experiments = {
    'free_delivery_threshold': {
        'name': 'Free Delivery Threshold',
        'baseline_rate': 0.28,
        'effect_size': 0.03,
        'n_users': 8600,
        'guardrails': ['aov', 'cancellation_rate']
    },
    'recommendation_algorithm': {
        'name': 'Recommendation Algorithm',
        'baseline_rate': 0.25,
        'effect_size': 0.02,
        'n_users': 20000,
        'guardrails': ['session_length', 'diversity']
    },
    'coupon_size': {
        'name': 'Coupon Size (Flat vs %)',
        'baseline_rate': 0.30,
        'effect_size': 0.04,
        'n_users': 12000,
        'guardrails': ['redemption_rate']
    },
    'checkout_ui': {
        'name': 'Checkout UI (Single vs Multi)',
        'baseline_rate': 0.65,
        'effect_size': 0.03,
        'n_users': 6000,
        'guardrails': ['payment_failure_rate']
    },
    'delivery_fee_structure': {
        'name': 'Delivery Fee Structure',
        'baseline_rate': 0.32,
        'effect_size': 0.02,
        'n_users': 10000,
        'guardrails': ['long_distance_orders']
    },
    'restaurant_ranking': {
        'name': 'Restaurant Ranking',
        'baseline_rate': 0.30,
        'effect_size': 0.015,
        'n_users': 20000,
        'guardrails': ['cancellation_rate']
    },
    'push_notification_timing': {
        'name': 'Push Notification Timing',
        'baseline_rate': 0.10,
        'effect_size': 0.03,
        'n_users': 10000,
        'guardrails': ['opt_out_rate']
    }
}

In [ ]:
print("\n📊 Generated experiment data for 7 A/B tests")

In [ ]:
# Create all experiment datasets
experiment_results = {}
for exp_key, exp_config in experiments.items():
    exp_data = create_experiment_data(
        exp_key,
        n_users=exp_config['n_users'],
        effect_size=exp_config['effect_size'],
        baseline_rate=exp_config['baseline_rate']
    )
    experiment_results[exp_key] = exp_data
    
    print(f"  ✅ {exp_config['name']}: {len(exp_data):,} users assigned")

---------------------------------------------------------------------
3. STATISTICAL ANALYSIS FUNCTIONS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("STATISTICAL ANALYSIS FUNCTIONS")
print("="*80)

In [ ]:
def calculate_sample_size(baseline_rate, mde, alpha=0.05, power=0.80, 
                         alternative='two-sided'):
    """
    Calculate required sample size for A/B test
    
    Parameters:
    - baseline_rate: Baseline conversion rate
    - mde: Minimum detectable effect (as proportion, e.g., 0.02 = 2%)
    - alpha: Significance level
    - power: Statistical power
    - alternative: 'two-sided' or 'one-sided'
    """
    
    # Standard normal quantiles
    z_alpha = norm.ppf(1 - alpha/2 if alternative == 'two-sided' else 1 - alpha)
    z_beta = norm.ppf(power)
    
    # Pooled proportion
    p_pooled = baseline_rate + mde / 2
    
    # Sample size per group
    n = (z_alpha * math.sqrt(2 * p_pooled * (1 - p_pooled)) + 
         z_beta * math.sqrt(baseline_rate * (1 - baseline_rate) + 
                            (baseline_rate + mde) * (1 - baseline_rate - mde))) ** 2 / (mde ** 2)
    
    return int(math.ceil(n))

In [ ]:
def two_proportion_test(control_success, control_n, treatment_success, treatment_n, 
                        alternative='two-sided', alpha=0.05):
    """
    Two-proportion z-test for comparing conversion rates
    """
    # Calculate proportions
    p1 = control_success / control_n if control_n > 0 else 0
    p2 = treatment_success / treatment_n if treatment_n > 0 else 0
    
    # Pooled proportion
    p_pooled = (control_success + treatment_success) / (control_n + treatment_n) if (control_n + treatment_n) > 0 else 0
    
    # Standard error
    se = math.sqrt(p_pooled * (1 - p_pooled) * (1/control_n + 1/treatment_n)) if control_n > 0 and treatment_n > 0 else 0
    
    # Z-score
    z_score = (p2 - p1) / se if se > 0 else 0
    
    # P-value
    if alternative == 'two-sided':
        p_value = 2 * (1 - norm.cdf(abs(z_score)))
    elif alternative == 'greater':
        p_value = 1 - norm.cdf(z_score)
    else:  # 'less'
        p_value = norm.cdf(z_score)
    
    # Confidence interval
    z_alpha = norm.ppf(1 - alpha/2 if alternative == 'two-sided' else 1 - alpha)
    se_diff = math.sqrt(p1 * (1 - p1) / control_n + p2 * (1 - p2) / treatment_n) if control_n > 0 and treatment_n > 0 else 0
    ci_lower = (p2 - p1) - z_alpha * se_diff
    ci_upper = (p2 - p1) + z_alpha * se_diff
    
    return {
        'control_rate': p1,
        'treatment_rate': p2,
        'difference': p2 - p1,
        'relative_lift': (p2 - p1) / p1 if p1 > 0 else 0,
        'z_score': z_score,
        'p_value': p_value,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        'significant': p_value < alpha
    }

In [ ]:
def t_test_metric(control_values, treatment_values, alternative='two-sided', alpha=0.05):
    """
    Welch's t-test for continuous metrics (e.g., AOV, session length)
    """
    t_stat, p_value = ttest_ind(treatment_values, control_values, equal_var=False, alternative=alternative)
    
    mean_diff = np.mean(treatment_values) - np.mean(control_values)
    relative_diff = mean_diff / np.mean(control_values) if np.mean(control_values) > 0 else 0
    
    return {
        'control_mean': np.mean(control_values),
        'treatment_mean': np.mean(treatment_values),
        'difference': mean_diff,
        'relative_lift': relative_diff,
        't_statistic': t_stat,
        'p_value': p_value,
        'significant': p_value < alpha
    }

In [ ]:
def calculate_power_achieved(n1, n2, p1, p2, alpha=0.05):
    """
    Calculate achieved power for a two-proportion test
    """
    # Pooled proportion
    p_pooled = (p1 * n1 + p2 * n2) / (n1 + n2) if (n1 + n2) > 0 else 0
    
    # Standard error under null
    se_null = math.sqrt(p_pooled * (1 - p_pooled) * (1/n1 + 1/n2)) if n1 > 0 and n2 > 0 else 0
    
    # Standard error under alternative
    se_alt = math.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2) if n1 > 0 and n2 > 0 else 0
    
    # Critical value
    z_alpha = norm.ppf(1 - alpha/2)
    
    # Power
    if se_null > 0 and se_alt > 0:
        power = norm.cdf((z_alpha * se_null - abs(p2 - p1)) / se_alt) + \
                1 - norm.cdf((-z_alpha * se_null - abs(p2 - p1)) / se_alt)
    else:
        power = 0
    
    return min(1, power)

---------------------------------------------------------------------
4. ANALYZE EACH EXPERIMENT
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("EXPERIMENT ANALYSIS")
print("="*80)

In [ ]:
def analyze_experiment(exp_data, exp_config, exp_key):
    """
    Comprehensive analysis of a single experiment
    """
    
    print(f"\n{'='*60}")
    print(f"📊 {exp_config['name']}")
    print(f"{'='*60}")
    
    # Split by variant
    control = exp_data[exp_data['variant'] == 'control']
    treatment = exp_data[exp_data['variant'] == 'treatment']
    
    n_control = len(control)
    n_treatment = len(treatment)
    
    # Primary metric: Conversion Rate
    control_conversions = control['converted'].sum()
    treatment_conversions = treatment['converted'].sum()
    
    # Analyze conversion
    conversion_results = two_proportion_test(
        control_conversions, n_control,
        treatment_conversions, n_treatment
    )
    
    print(f"\n📈 PRIMARY METRIC: Conversion Rate")
    print(f"  Control: {conversion_results['control_rate']*100:.2f}% ({control_conversions}/{n_control})")
    print(f"  Treatment: {conversion_results['treatment_rate']*100:.2f}% ({treatment_conversions}/{n_treatment})")
    print(f"  Absolute difference: {conversion_results['difference']*100:.2f}%")
    print(f"  Relative lift: {conversion_results['relative_lift']*100:.2f}%")
    print(f"  Z-score: {conversion_results['z_score']:.2f}")
    print(f"  P-value: {conversion_results['p_value']:.4f}")
    print(f"  95% CI: [{conversion_results['ci_lower']*100:.2f}%, {conversion_results['ci_upper']*100:.2f}%]")
    print(f"  Statistically significant: {'✅ YES' if conversion_results['significant'] else '❌ NO'}")
    
    # Calculate achieved power
    power_achieved = calculate_power_achieved(
        n_control, n_treatment,
        conversion_results['control_rate'],
        conversion_results['treatment_rate']
    )
    print(f"  Achieved power: {power_achieved*100:.1f}%")
    
    # Secondary metrics
    print(f"\n📊 SECONDARY METRICS:")
    
    # AOV
    control_aov = control[control['converted']]['order_value'].dropna()
    treatment_aov = treatment[treatment['converted']]['order_value'].dropna()
    
    if len(control_aov) > 0 and len(treatment_aov) > 0:
        aov_results = t_test_metric(control_aov, treatment_aov)
        print(f"  AOV:")
        print(f"    Control: ₹{aov_results['control_mean']:.2f}")
        print(f"    Treatment: ₹{aov_results['treatment_mean']:.2f}")
        print(f"    Difference: ₹{aov_results['difference']:.2f} ({aov_results['relative_lift']*100:.2f}%)")
        print(f"    Significant: {'✅' if aov_results['significant'] else '❌'}")
    
    # Cancellation Rate (guardrail)
    control_cancellations = control['cancelled'].sum()
    treatment_cancellations = treatment['cancelled'].sum()
    
    if n_control > 0 and n_treatment > 0:
        cancel_control_rate = control_cancellations / n_control
        cancel_treatment_rate = treatment_cancellations / n_treatment
        
        cancel_results = two_proportion_test(
            control_cancellations, n_control,
            treatment_cancellations, n_treatment
        )
        
        print(f"  Cancellation Rate:")
        print(f"    Control: {cancel_control_rate*100:.2f}%")
        print(f"    Treatment: {cancel_treatment_rate*100:.2f}%")
        print(f"    Difference: {cancel_results['difference']*100:.2f}%")
        print(f"    Significant: {'✅' if cancel_results['significant'] else '❌'}")
    
    # Payment Success Rate
    if 'payment_success' in control.columns:
        control_payment = control['payment_success'].sum()
        treatment_payment = treatment['payment_success'].sum()
        
        payment_results = two_proportion_test(
            control_payment, n_control,
            treatment_payment, n_treatment
        )
        
        print(f"  Payment Success Rate:")
        print(f"    Control: {payment_results['control_rate']*100:.2f}%")
        print(f"    Treatment: {payment_results['treatment_rate']*100:.2f}%")
        print(f"    Significant: {'✅' if payment_results['significant'] else '❌'}")
    
    # Business Decision
    print(f"\n💡 BUSINESS DECISION:")
    
    if conversion_results['significant']:
        if conversion_results['difference'] > 0:
            # Positive result - check guardrails
            if cancel_results['significant'] and cancel_results['difference'] > 0:
                print("  ⚠️ SHIP WITH CAUTION: Positive conversion lift but guardrail breach")
                print("  → Investigate root cause of increased cancellations")
                decision = 'HOLD'
            else:
                print("  ✅ SHIP: Statistically significant positive result with clean guardrails")
                decision = 'SHIP'
        else:
            print("  ❌ KILL: Statistically significant negative result")
            decision = 'KILL'
    else:
        print("  ⏸️ HOLD: Not statistically significant - need more data")
        decision = 'HOLD'
    
    return {
        'experiment': exp_key,
        'decision': decision,
        'conversion_results': conversion_results,
        'aov_results': aov_results if 'aov_results' in locals() else None,
        'cancel_results': cancel_results if 'cancel_results' in locals() else None
    }

In [ ]:
# Analyze all experiments
experiment_decisions = {}

In [ ]:
for exp_key, exp_config in experiments.items():
    exp_data = experiment_results[exp_key]
    results = analyze_experiment(exp_data, exp_config, exp_key)
    experiment_decisions[exp_key] = results

---------------------------------------------------------------------
5. EXPERIMENT DECISION SUMMARY
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("EXPERIMENT DECISION SUMMARY")
print("="*80)

In [ ]:
decision_summary = []
for exp_key, results in experiment_decisions.items():
    conv = results['conversion_results']
    decision_summary.append({
        'Experiment': experiments[exp_key]['name'],
        'Lift': f"{conv['relative_lift']*100:.2f}%",
        'P-Value': f"{conv['p_value']:.4f}",
        'Significant': '✅' if conv['significant'] else '❌',
        'Decision': results['decision']
    })

In [ ]:
decision_df = pd.DataFrame(decision_summary)
print("\n📊 Experiment Decision Summary:")
print(decision_df.to_string(index=False))

In [ ]:
# Visualize experiment results
fig, axes = plt.subplots(2, 1, figsize=(14, 12))
fig.suptitle('A/B Test Results Summary', fontsize=16, fontweight='bold')

In [ ]:
# 5.1 Conversion Lift with Confidence Intervals
ax = axes[0]

In [ ]:
experiment_names = [experiments[exp]['name'] for exp in experiment_decisions.keys()]
lifts = [experiment_decisions[exp]['conversion_results']['relative_lift'] * 100 for exp in experiment_decisions.keys()]
ci_lower = [experiment_decisions[exp]['conversion_results']['ci_lower'] * 100 for exp in experiment_decisions.keys()]
ci_upper = [experiment_decisions[exp]['conversion_results']['ci_upper'] * 100 for exp in experiment_decisions.keys()]
significant = [experiment_decisions[exp]['conversion_results']['significant'] for exp in experiment_decisions.keys()]

In [ ]:
x = np.arange(len(experiment_names))
colors = ['#2ecc71' if s else '#e74c3c' for s in significant]

In [ ]:
bars = ax.bar(x, lifts, color=colors, alpha=0.7)
ax.errorbar(x, lifts, yerr=[lifts - ci_lower, ci_upper - lifts], 
            fmt='none', ecolor='black', capsize=5, capthick=2)

In [ ]:
ax.axhline(y=0, color='black', linestyle='-', alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels(experiment_names, rotation=45, ha='right')
ax.set_ylabel('Relative Lift (%)')
ax.set_title('Experiment Results: Conversion Rate Lift with 95% CI')
ax.grid(True, alpha=0.3)

In [ ]:
# Add significance indicators
for i, bar in enumerate(bars):
    height = bar.get_height()
    sig_text = '✅' if significant[i] else '❌'
    ax.text(bar.get_x() + bar.get_width()/2, 
            height + (0.5 if height > 0 else -1.5), 
            sig_text, ha='center', va='bottom', fontsize=12)

In [ ]:
# 5.2 P-Value Heatmap
ax = axes[1]
p_values = [experiment_decisions[exp]['conversion_results']['p_value'] for exp in experiment_decisions.keys()]
p_value_matrix = np.array(p_values).reshape(1, -1)

In [ ]:
sns.heatmap(p_value_matrix, ax=ax, annot=True, fmt='.4f', cmap='RdYlGn_r',
            xticklabels=experiment_names, yticklabels=['P-Value'],
            cbar_kws={'label': 'P-Value', 'ticks': [0, 0.05, 0.1, 0.5, 1]})
ax.set_title('P-Values by Experiment')
ax.set_xlabel('Experiment')

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/experiment_summary.png', dpi=300, bbox_inches='tight')
plt.show()

---------------------------------------------------------------------
6. POWER ANALYSIS CALCULATOR
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("POWER ANALYSIS CALCULATOR")
print("="*80)

In [ ]:
def power_analysis_tool():
    """
    Interactive power analysis tool for experiment design
    """
    
    print("\n📊 POWER ANALYSIS CALCULATOR")
    print("="*50)
    
    # Common scenarios
    scenarios = [
        {'name': 'Small lift (2%)', 'baseline': 0.28, 'mde': 0.02, 'power': 0.80},
        {'name': 'Medium lift (3%)', 'baseline': 0.28, 'mde': 0.03, 'power': 0.80},
        {'name': 'Large lift (5%)', 'baseline': 0.28, 'mde': 0.05, 'power': 0.80},
        {'name': 'Checkout optimization (3%)', 'baseline': 0.65, 'mde': 0.03, 'power': 0.80},
        {'name': 'Rare event (5% lift)', 'baseline': 0.10, 'mde': 0.05, 'power': 0.80},
    ]
    
    print("\n📋 Sample Size Requirements for Common Scenarios:")
    print("="*60)
    
    for scenario in scenarios:
        n_per_group = calculate_sample_size(
            scenario['baseline'],
            scenario['mde'],
            power=scenario['power']
        )
        total_n = n_per_group * 2
        
        print(f"\n{scenario['name']}:")
        print(f"  Baseline rate: {scenario['baseline']*100:.0f}%")
        print(f"  MDE: {scenario['mde']*100:.0f}%")
        print(f"  Required per group: {n_per_group:,}")
        print(f"  Total sample size: {total_n:,}")
    
    return scenarios

In [ ]:
power_scenarios = power_analysis_tool()

In [ ]:
# Visualize power analysis
fig, ax = plt.subplots(figsize=(12, 8))

In [ ]:
# Create grid of sample sizes for different baseline rates and MDEs
baseline_rates = np.linspace(0.05, 0.70, 10)
mdes = np.linspace(0.01, 0.10, 10)

In [ ]:
sample_size_grid = np.zeros((len(baseline_rates), len(mdes)))

In [ ]:
for i, baseline in enumerate(baseline_rates):
    for j, mde in enumerate(mdes):
        sample_size_grid[i, j] = calculate_sample_size(baseline, mde, power=0.80)

In [ ]:
# Create heatmap
im = ax.imshow(sample_size_grid, cmap='YlOrRd', aspect='auto', origin='lower',
               extent=[mdes.min()*100, mdes.max()*100, baseline_rates.min()*100, baseline_rates.max()*100])

In [ ]:
ax.set_xlabel('Minimum Detectable Effect (%)')
ax.set_ylabel('Baseline Conversion Rate (%)')
ax.set_title('Sample Size per Group (80% Power, α=0.05)')
plt.colorbar(im, ax=ax, label='Sample Size per Group')

In [ ]:
# Add contour lines
contour_levels = [500, 1000, 2000, 5000, 10000, 20000, 50000]
CS = ax.contour(mdes*100, baseline_rates*100, sample_size_grid, 
                levels=contour_levels, colors='black', alpha=0.5)
ax.clabel(CS, inline=True, fontsize=8)

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/power_analysis_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

---------------------------------------------------------------------
7. SEQUENTIAL TESTING SIMULATION
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("SEQUENTIAL TESTING SIMULATION")
print("="*80)

In [ ]:
def simulate_sequential_test(n_total=10000, true_effect=0.02, baseline_rate=0.28, 
                            alpha=0.05, look_every=100):
    """
    Simulate sequential testing to show peeking problem
    """
    
    np.random.seed(42)
    
    # Generate data
    n_control = n_total // 2
    n_treatment = n_total - n_control
    
    # Control group
    control_converted = np.random.binomial(1, baseline_rate, n_control)
    
    # Treatment group
    treatment_converted = np.random.binomial(1, baseline_rate + true_effect, n_treatment)
    
    # Sequential analysis
    results = []
    p_values = []
    cumulative_diff = []
    
    for i in range(look_every, n_total // 2 + 1, look_every):
        # Use first i observations from each group
        control_success = control_converted[:i].sum()
        treatment_success = treatment_converted[:i].sum()
        
        # Calculate z-test
        result = two_proportion_test(control_success, i, treatment_success, i)
        
        results.append({
            'n': i * 2,
            'p_value': result['p_value'],
            'diff': result['difference'],
            'significant': result['significant']
        })
        
        p_values.append(result['p_value'])
        cumulative_diff.append(result['difference'])
    
    return pd.DataFrame(results)

In [ ]:
# Run sequential test simulation
sequential_results = simulate_sequential_test(n_total=5000, true_effect=0.02)

In [ ]:
# Visualize
fig, axes = plt.subplots(2, 1, figsize=(12, 10))
fig.suptitle('Sequential Testing Simulation (P-Peeking Problem)', fontsize=14, fontweight='bold')

In [ ]:
# P-values over time
ax = axes[0]
ax.plot(sequential_results['n'], sequential_results['p_value'], 
        marker='o', linewidth=2, color='#3498db')
ax.axhline(y=0.05, color='red', linestyle='--', label='α = 0.05')
ax.axhline(y=0.01, color='orange', linestyle='--', label='α = 0.01')
ax.set_xlabel('Sample Size (Total)')
ax.set_ylabel('P-Value')
ax.set_title('P-Values Over Time (Sequential Testing)')
ax.legend()
ax.grid(True, alpha=0.3)

In [ ]:
# Cumulative effect
ax = axes[1]
ax.plot(sequential_results['n'], sequential_results['diff'] * 100, 
        marker='o', linewidth=2, color='#2ecc71')
ax.axhline(y=0, color='black', linestyle='-', alpha=0.5)
ax.axhline(y=2, color='green', linestyle='--', label='True Effect (2%)')
ax.set_xlabel('Sample Size (Total)')
ax.set_ylabel('Observed Lift (%)')
ax.set_title('Cumulative Observed Effect Over Time')
ax.legend()
ax.grid(True, alpha=0.3)

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/sequential_testing.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
print("\n⚠️ KEY INSIGHT: Early peeking at p-values can lead to false positives")
print("  • P-value fluctuates significantly with small sample sizes")
print("  • Stabilizes around the true effect only after sufficient samples")
print("  • Recommendation: Pre-register sample size and don't peek early")

---------------------------------------------------------------------
8. GUARDRAIL ANALYSIS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("GUARDRAIL ANALYSIS")
print("="*80)

In [ ]:
# Analyze guardrail metrics for each experiment
guardrail_summary = []

In [ ]:
for exp_key, exp_config in experiments.items():
    exp_data = experiment_results[exp_key]
    
    control = exp_data[exp_data['variant'] == 'control']
    treatment = exp_data[exp_data['variant'] == 'treatment']
    
    # Cancellation rate
    control_cancel = control['cancelled'].mean()
    treatment_cancel = treatment['cancelled'].mean()
    
    cancel_diff = treatment_cancel - control_cancel
    cancel_sig = two_proportion_test(
        control['cancelled'].sum(), len(control),
        treatment['cancelled'].sum(), len(treatment)
    )['significant']
    
    # Payment success rate
    if 'payment_success' in control.columns:
        control_payment = control['payment_success'].mean()
        treatment_payment = treatment['payment_success'].mean()
        payment_diff = treatment_payment - control_payment
        payment_sig = two_proportion_test(
            control['payment_success'].sum(), len(control),
            treatment['payment_success'].sum(), len(treatment)
        )['significant']
    else:
        control_payment = treatment_payment = payment_diff = payment_sig = None
    
    guardrail_summary.append({
        'Experiment': exp_config['name'],
        'Cancel Control': f"{control_cancel*100:.2f}%",
        'Cancel Treatment': f"{treatment_cancel*100:.2f}%",
        'Cancel Diff': f"{cancel_diff*100:.2f}%",
        'Cancel Significant': '✅' if cancel_sig else '❌',
        'Payment Control': f"{control_payment*100:.2f}%" if control_payment is not None else 'N/A',
        'Payment Treatment': f"{treatment_payment*100:.2f}%" if treatment_payment is not None else 'N/A',
        'Payment Diff': f"{payment_diff*100:.2f}%" if payment_diff is not None else 'N/A'
    })

In [ ]:
guardrail_df = pd.DataFrame(guardrail_summary)
print("\n📊 Guardrail Metrics Summary:")
print(guardrail_df.to_string(index=False))

---------------------------------------------------------------------
9. BUSINESS RECOMMENDATIONS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("BUSINESS RECOMMENDATIONS")
print("="*80)

In [ ]:
# Compile recommendations
recommendations = []

In [ ]:
for exp_key, results in experiment_decisions.items():
    exp_config = experiments[exp_key]
    conv = results['conversion_results']
    
    if results['decision'] == 'SHIP':
        priority = 'P0'
        action = 'Ship experiment'
        expected_impact = f"{conv['relative_lift']*100:.1f}% lift in conversion"
    elif results['decision'] == 'HOLD':
        if conv['significant']:
            priority = 'P1'
            action = 'Investigate guardrail breach before shipping'
            expected_impact = f"Potential {conv['relative_lift']*100:.1f}% lift but guardrail concerns"
        else:
            priority = 'P2'
            action = 'Run longer or increase sample size'
            expected_impact = 'Insufficient evidence'
    else:  # KILL
        priority = 'P3'
        action = 'Kill experiment'
        expected_impact = 'Negative or negligible impact'
    
    recommendations.append({
        'Experiment': exp_config['name'],
        'Decision': results['decision'],
        'Priority': priority,
        'Action': action,
        'Expected Impact': expected_impact,
        'Confidence': 'High' if conv['p_value'] < 0.01 else 'Medium' if conv['p_value'] < 0.05 else 'Low'
    })

In [ ]:
recommendations_df = pd.DataFrame(recommendations)
print("\n📋 Experiment Recommendations:")
print(recommendations_df.to_string(index=False))

---------------------------------------------------------------------
10. EXPORT RESULTS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("EXPORTING RESULTS")
print("="*80)

In [ ]:
# Create experiment summary for each test
for exp_key, exp_data in experiment_results.items():
    exp_data.to_csv(f'../outputs/cleaned_data/experiment_{exp_key}.csv', index=False)

In [ ]:
# Save recommendations
recommendations_df.to_csv('../outputs/cleaned_data/experiment_recommendations.csv', index